> **GitHub管理版（2026-11-05移管）**
> 既存の実験コードを移管したものです。セル出力は除去しており、今回の移管では実験を再実行していません。入力データ・保存済み結果は従来のDrive等を参照します。実行条件、認証、保存先を確認してから実行してください。元ファイルと用途は [ノートブック一覧](https://github.com/ryy1210/RMT_utils/blob/main/notebooks/README.md) を参照してください。

# 10 — DEによるbulkKS変化とPPL改善の関係
保存済み08_2・実験2の単独行列LRAを分析する。**モデル取得・SVD・DE・推論・LoRAは実行しない。CPUで「すべて実行」可能。**

主解析：preDE（BEMA分散）→ DE後（分散1）、matched_upperの両端bulkKS。DE間で保持rankが同じことを検証する。
- ΔKS = post − pre：正ならMP適合悪化。
- PPL改善率 = 100 × (1 − PPL_DE/PPL_off)：正ならDE優位。
- 正の関連は「KS悪化とPPL改善」、負の関連は「KS改善とPPL改善」に対応。

結果は関連を示すもので、追加学習の進行やMP閾値の理論的妥当性を証明しない。実験2のDE後rankもpreDE閾値由来である。両端bulkKSは前後で対象集合が変わるため、除外率と上端bulkKSを併記する。

必要ライブラリ：numpy, pandas, scipy, matplotlib。未導入なら利用するカーネルにインストールする。ColabではCPUを選び、Drive未マウント時は接続を許可する。

In [ ]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import matplotlib.pyplot as plt
from IPython.display import display

# 任意：保存先が異なる場合のみ指定。通常は変更不要。
DATA_ROOT_OVERRIDE = None
candidates = [Path('/content/drive/MyDrive/TUS/hashiguchi/data'),
    Path.home()/'Library/CloudStorage/GoogleDrive-ryoya.zushi1210@gmail.com/マイドライブ/TUS/hashiguchi/data']
if Path('/content').exists() and not candidates[0].exists():
    from google.colab import drive
    drive.mount('/content/drive')
DATA_ROOT = Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else next((p for p in candidates if p.exists()), None)
if DATA_ROOT is None: raise FileNotFoundError('DATA_ROOT_OVERRIDEにhashiguchi/dataの場所を指定してください')
SOURCE = DATA_ROOT/'ibis2026_de_lra/ibis2026-de-controlled-v2-bulk-order'
KS_PATH = DATA_ROOT/'bulk_ks_two_sided/two-sided-bulk-ks-v1/two_sided_bulk_ks.csv'
OUTPUT = DATA_ROOT/'ibis2026_de_lra/bulkks-ppl-association-v1'
OUTPUT.mkdir(parents=True, exist_ok=True)
BOOTSTRAPS = 1000
SEED = 20261023
KS_EPS = 0.005  # 実用的な変化の区分用。検定の有意水準ではない。
PPL_EPS = 1.0   # 改善率のpercentage points。0/0でも感度分析する。
MODULES = ['up_proj','down_proj','gate_proj','q_proj','k_proj','v_proj','o_proj']
COLORS = dict(zip(MODULES, plt.get_cmap('tab10').colors[:7]))
plt.rcParams.update({'figure.dpi':120, 'axes.grid':True, 'grid.alpha':0.2})
def save_table(df,name):
    df.to_csv(OUTPUT/(name+'.csv'),index=False)
def save_fig(fig,name):
    fig.tight_layout();fig.savefig(OUTPUT/(name+'.png'),bbox_inches='tight')
    fig.savefig(OUTPUT/(name+'.pdf'),bbox_inches='tight');plt.show();plt.close(fig)
def canonical(s): return s.removeprefix('model.')
print('Input:',SOURCE, '\nOutput:',OUTPUT)


## 1. 入力と対応関係の検証
08_2の集計とrank_planを読む。08から再利用された結果は08_2側に含まれるため、08を追加結合して二重計上しない。欠損・重複・異なるrank・異なる形状があれば停止する。入力ファイルのSHA256を記録するが、元の全lossesファイルの再監査ではない。

In [ ]:
ppl = pd.read_csv(SOURCE/'paired_ppl.csv')
ppl = ppl.loc[(ppl.experiment=='E2') & (ppl.variant=='all_single')].copy()
ppl['name'] = ppl.scope.map(canonical)
assert not ppl.duplicated(['model','name']).any()
assert np.isfinite(ppl[['ppl_off','ppl_de','baseline_ppl']]).all().all()
assert (ppl[['ppl_off','ppl_de','baseline_ppl']]>0).all().all()
assert np.allclose(ppl.reduction_off,ppl.reduction_de,rtol=0,atol=1e-12)
plans = json.loads((SOURCE/'rank_plan.json').read_text())
rank_rows=[]
for j in plans:
    if j['experiment']=='E2' and j['variant']=='all_single':
        assert len(j['names'])==1
        name=j['names'][0]
        rank_rows.append(dict(model=j['model'],name=canonical(name),de=bool(j['use_de']),rank=j['ranks'][name]))
r=pd.DataFrame(rank_rows)
assert not r.duplicated(['model','name','de']).any()
r=r.pivot(index=['model','name'],columns='de',values='rank')
assert {False,True} <= set(r.columns) and r.notna().all().all()
assert (r[False]==r[True]).all()
r=r[False].rename('rank').reset_index()
df=ppl.merge(r,on=['model','name'],how='left',validate='one_to_one')
ks=pd.read_csv(KS_PATH)
ks=ks.loc[(ks.method=='matched_upper') & ks.condition.isin(['preDE','DE_1'])].copy()
ks['name']=ks.name.map(canonical)
assert not ks.duplicated(['model','name','condition']).any()
assert ks.status.eq('ok').all()
fields=['two_sided_bulkKS','upper_only_bulkKS','fullKS','low_ratio','high_ratio','bulk_ratio','p','n','block_index','module_type']
for condition,suffix in [('preDE','pre'),('DE_1','post')]:
    part=ks.loc[ks.condition==condition,['model','name']+fields]
    assert set(map(tuple,part[['model','name']].values))==set(map(tuple,df[['model','name']].values))
    df=df.merge(part.rename(columns={c:c+'_'+suffix for c in fields}),on=['model','name'],how='left',validate='one_to_one')
assert not df.isna()[['rank']+[c+'_'+s for c in fields for s in ['pre','post']]].any().any()
for c in ['p','n','block_index','module_type']: assert (df[c+'_pre']==df[c+'_post']).all()
assert df.groupby('model').size().to_dict()=={'gemma':238,'llama':196}
df['module']=df.module_type_pre;df['block']=df.block_index_pre.astype(int)
df['rank_fraction']=df['rank']/df.p_pre
df['gamma']=df.p_pre/df.n_pre
assert df.rank_fraction.between(0,1).all()
df['depth']=df.groupby('model').block.transform(lambda s:s/s.max())
df['delta_ks']=df.two_sided_bulkKS_post-df.two_sided_bulkKS_pre
df['delta_upper_ks']=df.upper_only_bulkKS_post-df.upper_only_bulkKS_pre
df['delta_full_ks']=df.fullKS_post-df.fullKS_pre
for edge in ['low','high','bulk']: df['delta_'+edge+'_ratio']=df[edge+'_ratio_post']-df[edge+'_ratio_pre']
df['gain']=100*(1-df.ppl_de/df.ppl_off)
df['ppl_gap']=df.ppl_off-df.ppl_de
assert np.allclose(df.gain,df.improvement_pct)
inputs=[SOURCE/'paired_ppl.csv',SOURCE/'rank_plan.json',SOURCE/'manifest.json',KS_PATH]
receipt={'inputs':{str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs},
 'rows':df.groupby('model').size().to_dict(),'bootstrap':BOOTSTRAPS,'seed':SEED,'ks_eps':KS_EPS,'ppl_eps':PPL_EPS,
 'python':platform.python_version(),'limitations':'Saved aggregates checked; raw loss files not re-audited.'}
(OUTPUT/'input_audit.json').write_text(json.dumps(receipt,indent=2,ensure_ascii=False))
save_table(df,'joined_matrix_data')
display(df.groupby('model').agg(matrices=('name','size'),median_gain=('gain','median'),median_delta_KS=('delta_ks','median')))


## 2. 相関と不確実性
Spearman順位相関をモデル別・モジュール別に計算する。95%区間は**Transformerブロック単位のbootstrap**（モデル内で復元抽出）。ブロック間の独立性も近似であり、別モデル・別学習seedへの一般化を保証しない。多重な探索なので、有意な箇所だけを抜き出さない。

In [ ]:
def corr(x,y):
    x=np.asarray(x,float);y=np.asarray(y,float)
    if len(x)<4 or np.std(x)==0 or np.std(y)==0:return np.nan
    return float(np.corrcoef(x,y)[0,1])
def spearman(g,x='delta_ks',y='gain'):
    return corr(rankdata(g[x]),rankdata(g[y]))
def cluster_ci(g,stat,seed=SEED):
    rng=np.random.default_rng(seed)
    groups=[v for _,v in g.groupby('block')]
    vals=[]
    for _ in range(BOOTSTRAPS):
        sample=pd.concat([groups[i] for i in rng.integers(0,len(groups),len(groups))],ignore_index=True)
        vals.append(stat(sample))
    vals=np.asarray(vals);vals=vals[np.isfinite(vals)]
    return (*np.quantile(vals,[.025,.975]),len(vals)) if len(vals) else (np.nan,np.nan,0)
rows=[]
for model,g in df.groupby('model'):
    for module,h in [('ALL',g)]+list(g.groupby('module')):
        lo,hi,valid=cluster_ci(h,spearman)
        rows.append(dict(model=model,module=module,n=len(h),rho=spearman(h),ci_low=lo,ci_high=hi,valid_bootstraps=valid))
correlations=pd.DataFrame(rows);save_table(correlations,'spearman_by_module');display(correlations.round(4))
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,(model,g) in zip(axes,df.groupby('model')):
    for module,h in g.groupby('module'):ax.scatter(h.delta_ks,h.gain,label=module,color=COLORS[module],s=22,alpha=.7)
    ax.axvline(0,color='black',lw=.8);ax.axhline(0,color='black',lw=.8)
    ax.set(title=f'{model}: all matrices',xlabel='Delta two-sided bulkKS (post - pre)',ylabel='PPL improvement (%)')
axes[-1].legend(fontsize=8);save_fig(fig,'main_scatter')
for model,g in df.groupby('model'):
    fig,axes=plt.subplots(2,4,figsize=(16,8))
    for ax,module in zip(axes.flat,MODULES):
        h=g[g.module==module];ax.scatter(h.delta_ks,h.gain,c=h.depth,cmap='viridis',vmin=0,vmax=1,s=25)
        ax.axvline(0,color='black',lw=.6);ax.axhline(0,color='black',lw=.6)
        ax.set(title=f'{module}: rho={spearman(h):.2f}',xlabel='Delta bulkKS',ylabel='PPL improvement (%)')
    axes.flat[-1].axis('off');fig.suptitle(model+' | color: shallow (purple) to deep (yellow)')
    save_fig(fig,'module_scatter_'+model)


## 3. 改善・悪化の組合せとPPL悪化例
区分の閾値は説明用で、統計的な有意差ではない。既存のPPL区間がある場合は併記する。初期区分とゼロ基準の両方を保存する。

In [ ]:
def direction(v,eps,positive,negative):
    return np.where(v>eps,positive,np.where(v < -eps,negative,'near_zero'))
for ke,pe,label in [(KS_EPS,PPL_EPS,'practical'),(0,0,'zero')]:
    q=df.copy();q['KS_change']=direction(q.delta_ks,ke,'worse','better');q['PPL_change']=direction(q.gain,pe,'better','worse')
    table=q.groupby(['model','KS_change','PPL_change']).agg(n=('name','size'),median_gain=('gain','median'),median_delta_KS=('delta_ks','median')).reset_index()
    save_table(table,'quadrants_'+label);print(label);display(table)
    if label=='practical':classified=q
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,(model,g) in zip(axes,classified.groupby('model')):
    cats=['better','near_zero','worse'];values=[g.loc[g.PPL_change==c,'delta_ks'].values for c in cats]
    ax.boxplot(values,tick_labels=cats,showfliers=True);ax.axhline(0,c='black',lw=.7)
    ax.set(title=model,xlabel='PPL change after DE',ylabel='Delta two-sided bulkKS')
save_fig(fig,'ks_by_ppl_change')
cols=['model','name','module','block','ppl_off','ppl_de','gain','delta_ks','two_sided_bulkKS_pre','two_sided_bulkKS_post','rank_fraction','delta_low_ratio','delta_high_ratio','bulk_ratio_pre','bulk_ratio_post','ci_low_pct','ci_high_pct']
worst=df.sort_values('gain').groupby('model',group_keys=False).head(15)[cols]
save_table(worst,'ppl_worsened_examples');display(worst)


## 4. モジュール・深さ・保持rank比を考慮した関連
主相関に加え、各変数を順位化し、モジュールのダミー変数・深さ・保持rank比・preDE bulkKSを線形に除いた残差同士を相関させる（調整済み順位相関）。これは因果効果の推定ではない。

形状比はモデル内でモジュールと強く対応するため、モジュールと同時に無条件で追加しない。preDE値での調整には測定誤差・差分との数学的結合の限界があるので、未調整値と並べて解釈する。

In [ ]:
def adjusted(g):
    controls=np.column_stack([np.ones(len(g)),pd.get_dummies(g.module,drop_first=True,dtype=float).to_numpy(),
        *[rankdata(g[c]) for c in ['depth','rank_fraction','two_sided_bulkKS_pre']]])
    # rank-deficientな場合にも最小二乗の射影として扱う。
    x=rankdata(g.delta_ks);y=rankdata(g.gain)
    rx=x-controls@np.linalg.lstsq(controls,x,rcond=None)[0]
    ry=y-controls@np.linalg.lstsq(controls,y,rcond=None)[0]
    return corr(rx,ry)
rows=[]
for model,g in df.groupby('model'):
    lo,hi,valid=cluster_ci(g,adjusted)
    rows.append(dict(model=model,raw_rho=spearman(g),adjusted_rank_correlation=adjusted(g),ci_low=lo,ci_high=hi,valid_bootstraps=valid))
adjusted_results=pd.DataFrame(rows);save_table(adjusted_results,'adjusted_association');display(adjusted_results)


## 5. 補助分析：bulk選択と指標への依存
全体KS・上端bulkKS・除外率の変化を同じPPL改善率と比較する。両端bulkKSだけの改善を「ノイズ判別の成功」と解釈しない。PPL差でも関連の方向を確認するが、モデル間でPPLの絶対値を直接比較しない。

In [ ]:
metrics=['delta_ks','delta_upper_ks','delta_full_ks','delta_low_ratio','delta_high_ratio','delta_bulk_ratio']
rows=[]
for model,g in df.groupby('model'):
    for metric in metrics:
        stat=lambda h,m=metric:spearman(h,x=m)
        lo,hi,valid=cluster_ci(g,stat)
        rows.append(dict(model=model,metric=metric,rho=stat(g),ci_low=lo,ci_high=hi,valid_bootstraps=valid))
sensitivity=pd.DataFrame(rows);save_table(sensitivity,'auxiliary_associations');display(sensitivity)
for model,g in df.groupby('model'):
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax,metric in zip(axes,['delta_upper_ks','delta_low_ratio','delta_high_ratio']):
        for module,h in g.groupby('module'):ax.scatter(h[metric],h.gain,color=COLORS[module],s=18,alpha=.6)
        ax.axvline(0,c='black',lw=.6);ax.axhline(0,c='black',lw=.6)
        ax.set(xlabel=metric,ylabel='PPL improvement (%)',title=model)
    save_fig(fig,'auxiliary_scatter_'+model)
absolute=pd.DataFrame([dict(model=m,rho_gain=spearman(g),rho_absolute_ppl_gap=spearman(g,y='ppl_gap')) for m,g in df.groupby('model')])
save_table(absolute,'ppl_metric_sensitivity');display(absolute)
mdlines=['# bulkKSとPPL改善の関連：探索的集計','', '正のrho: KS悪化とPPL改善が対応。負のrho: KS改善とPPL改善が対応。', '', '因果・学習の進行・MP閾値の正しさは立証しない。']
for row in correlations[correlations.module=='ALL'].itertuples():
    mdlines.append(f'- {row.model}: n={row.n}, Spearman rho={row.rho:.3f}, block bootstrap 95% interval [{row.ci_low:.3f}, {row.ci_high:.3f}]')
mdlines+=['','詳細はspearman_by_module.csv、adjusted_association.csv、auxiliary_associations.csvを参照。','PPL区間は既存評価から継承。ブロックbootstrap区間は別の不確実性であり混同しない。']
(OUTPUT/'summary.md').write_text('\n'.join(mdlines),encoding='utf-8')
print('保存完了:',OUTPUT)
